#Week 7: Data Pipelines & Automation

##Project Overview

This project focuses on building a simple ETL (Extract, Transform, Load) pipeline using real-time weather data from the OpenWeather API.

The objective is to extract weather information for three cities, transform and clean the data using Python and Pandas, store the processed data in a CSV file, and perform basic analysis.

###1. Extract Data

Objective: Retrieve real-time weather data from the OpenWeather API for the selected cities.

In [1]:
import requests
import pandas as pd

In [2]:
API_KEY = ""

In [3]:
city = "Djibouti"

url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": city,
    "appid": API_KEY,
    "units": "metric"
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.json())

200
{'coord': {'lon': 42.5, 'lat': 11.5}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'base': 'stations', 'main': {'temp': 38.04, 'feels_like': 36.71, 'temp_min': 38.04, 'temp_max': 38.04, 'pressure': 1004, 'humidity': 20, 'sea_level': 1004, 'grnd_level': 939}, 'visibility': 10000, 'wind': {'speed': 0.8, 'deg': 50, 'gust': 1.73}, 'clouds': {'all': 9}, 'dt': 1787756002, 'sys': {'country': 'DJ', 'sunrise': 1787713203, 'sunset': 1787757852}, 'timezone': 10800, 'id': 223816, 'name': 'Djibouti', 'cod': 200}


In [6]:
data = response.json()

weather_data = {
    "City": data["name"],
    "Temperature": data["main"]["temp"],
    "Humidity": data["main"]["humidity"],
    "Weather Condition": data["weather"][0]["description"],
    "Wind Speed": data["wind"]["speed"],
    "Date and Time": pd.to_datetime(data["dt"], unit="s")
}

print(weather_data)

{'City': 'Istanbul', 'Temperature': 28.3, 'Humidity': 88, 'Weather Condition': 'clear sky', 'Wind Speed': 7.72, 'Date and Time': Timestamp('2026-08-26 15:03:11')}


In [7]:
cities = ["Djibouti", "Paris", "Istanbul"]

weather_list = []

for city in cities:

    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(url, params=params)
    data = response.json()

    weather_data = {
        "City": data["name"],
        "Temperature": data["main"]["temp"],
        "Humidity": data["main"]["humidity"],
        "Weather Condition": data["weather"][0]["description"],
        "Wind Speed": data["wind"]["speed"],
        "Date and Time": pd.to_datetime(data["dt"], unit="s")
    }

    weather_list.append(weather_data)

df_weather = pd.DataFrame(weather_list)

df_weather

,City,Temperature,Humidity,Weather Condition,Wind Speed,Date and Time
0,Djibouti,38.04,20,clear sky,0.80,2026-08-26 15:08:16
1,Paris,28.29,46,overcast clouds,3.09,2026-08-26 14:59:58
2,Istanbul,28.30,88,clear sky,7.72,2026-08-26 15:03:32


## 2. Transform Data

In this step, the extracted weather data is cleaned and organized using Pandas.

The transformation process includes checking the dataset structure, identifying missing values, checking data types, and rounding numerical values.

In [8]:
df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   City               3 non-null      object        
 1   Temperature        3 non-null      float64       
 2   Humidity           3 non-null      int64         
 3   Weather Condition  3 non-null      object        
 4   Wind Speed         3 non-null      float64       
 5   Date and Time      3 non-null      datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(2)
memory usage: 276.0+ bytes


In [9]:
df_weather.isnull().sum()

,0
City,0
Temperature,0
Humidity,0
Weather Condition,0
Wind Speed,0
Date and Time,0


In [10]:
df_weather.dtypes

,0
City,object
Temperature,float64
Humidity,int64
Weather Condition,object
Wind Speed,float64
Date and Time,datetime64[ns]


In [11]:
df_weather["Temperature (°C)"] = df_weather["Temperature (°C)"].round(2)

df_weather["Wind Speed (m/s)"] = df_weather["Wind Speed (m/s)"].round(2)

KeyError: 'Temperature (°C)'

In [12]:
df_weather.columns

Index(['City', 'Temperature', 'Humidity', 'Weather Condition', 'Wind Speed',
       'Date and Time'],
      dtype='object')

In [13]:
df_weather["Temperature"] = df_weather["Temperature"].round(2)
df_weather["Wind Speed"] = df_weather["Wind Speed"].round(2)

In [14]:
df_weather

,City,Temperature,Humidity,Weather Condition,Wind Speed,Date and Time
0,Djibouti,38.04,20,clear sky,0.80,2026-08-26 15:08:16
1,Paris,28.29,46,overcast clouds,3.09,2026-08-26 14:59:58
2,Istanbul,28.30,88,clear sky,7.72,2026-08-26 15:03:32


## 3. Load Data

In this step, the transformed weather dataset is saved as a CSV file for future analysis and reuse.

In [15]:
df_weather.to_csv("weather_data.csv", index=False)

In [16]:
import os

print(os.path.exists("weather_data.csv"))

True


## 4. Basic Analysis

In this step, the cleaned weather data is analyzed to compare temperatures, humidity levels, and weather conditions across the three cities.

In [17]:
# Compare temperatures across cities

print("Temperature by city:")
print(df_weather[["City", "Temperature"]])

Temperature by city:
       City  Temperature
0  Djibouti        38.04
1     Paris        28.29
2  Istanbul        28.30


In [18]:
# City with the highest temperature

hottest_city = df_weather.loc[df_weather["Temperature"].idxmax()]

print("The city with the highest temperature is:",
      hottest_city["City"],
      "with",
      hottest_city["Temperature"],
      "°C")

The city with the highest temperature is: Djibouti with 38.04 °C


In [19]:
# City with the highest humidity

most_humid_city = df_weather.loc[df_weather["Humidity"].idxmax()]

print("The city with the highest humidity is:",
      most_humid_city["City"],
      "with",
      most_humid_city["Humidity"],
      "% humidity")

The city with the highest humidity is: Istanbul with 88 % humidity


In [20]:
# Compare weather conditions

print("Weather conditions by city:")

for index, row in df_weather.iterrows():
    print(row["City"], ":", row["Weather Condition"])

Weather conditions by city:
Djibouti : clear sky
Paris : overcast clouds
Istanbul : clear sky


#Conclusion :
## Djibouti recorded the highest temperature at 38.04°C, while Istanbul had the highest humidity at 88%. Djibouti and Istanbul had clear skies, whereas Paris had overcast clouds.